# Chapter 3 — Labels and agreement

Cohen's kappa from the 2×2 tables, the kappa paradox, Fleiss' kappa and Krippendorff's alpha on the
published examples, the three-rater worked example, and the sampling spread of kappa. Every cell
**asserts** what it checks.

In [ ]:
import math, os, sys, random
sys.path.insert(0, os.path.abspath(".."))      # run from notebooks/
sys.path.insert(0, os.path.abspath("."))       # or from the repo root
import numpy as np
from evalkit import agreement as A

## 3.2 — kappa, and the paradox

In [ ]:
for t in [(70, 10, 5, 15), (45, 5, 5, 45), (90, 4, 4, 2), (92, 3, 3, 2)]:
    r = A.kappa_from_table(*t)
    print(t, f"p_o={r['po']:.2f}  p_e={r['pe']:.3f}  kappa={r['kappa']:.3f}")
assert round(A.kappa_from_table(70, 10, 5, 15)["kappa"], 3) == 0.571
assert round(A.kappa_from_table(90, 4, 4, 2)["kappa"], 3) == 0.291     # 92% agreement, kappa 0.29
a, b = A.table_to_labels(70, 10, 5, 15)
assert round(A.cohen_kappa(a, b), 3) == 0.571

## 3.3 — the published examples (Krippendorff 2011; the standard Fleiss example)

In [ ]:
N = None
units = [list(t) for t in zip([1, 2, 3, 3, 2, 1, 4, 1, 2, N, N, N], [1, 2, 3, 3, 2, 2, 4, 1, 2, 5, N, 3],
                              [N, 3, 3, 3, 2, 3, 4, 2, 2, 5, 1, N], [1, 2, 3, 3, 2, 4, 4, 1, 2, 5, 1, N])]
print("alpha:", round(A.krippendorff_alpha_nominal(units), 3))
assert round(A.krippendorff_alpha_nominal(units), 3) == 0.743
fl = [[0, 0, 0, 0, 14], [0, 2, 6, 4, 2], [0, 0, 3, 5, 6], [0, 3, 9, 2, 0], [2, 2, 8, 1, 1],
      [7, 7, 0, 0, 0], [3, 2, 6, 3, 0], [2, 5, 3, 2, 2], [6, 5, 2, 1, 0], [0, 2, 2, 3, 7]]
print("Fleiss:", round(A.fleiss_kappa(fl), 3))
assert round(A.fleiss_kappa(fl), 3) == 0.210

## 3.3 — three raters, ten items: find the rater who drifted

In [ ]:
THREE = [[1, 1, 1], [1, 1, 1], [1, 1, 0], [1, 1, 1], [0, 0, 0], [1, 0, 0], [1, 1, 1], [1, 1, 0], [0, 0, 0], [1, 1, 1]]
cnt = [[sum(x == 1 for x in u), sum(x == 0 for x in u)] for u in THREE]
lab = {r: [u[i] for u in THREE] for i, r in enumerate("ABC")}
print("Fleiss", round(A.fleiss_kappa(cnt), 3), " alpha", round(A.krippendorff_alpha_nominal(THREE), 3))
print({k: round(v, 3) for k, v in A.pairwise_cohen(lab).items()})
assert round(A.pairwise_cohen(lab)[("A", "C")], 3) == 0.4
gaps = [list(u) for u in THREE]; gaps[1][2] = gaps[6][1] = gaps[8][2] = None
assert round(A.krippendorff_alpha_nominal(gaps), 3) == 0.519

## Exercises — kappa by hand, and kappa's sampling spread

In [ ]:
assert round(A.kappa_from_table(120, 20, 10, 50)["kappa"], 3) == 0.659
assert round(A.kappa_from_table(165, 20, 10, 5)["kappa"], 2) == 0.17
for n in (50, 100, 200, 400):
    ks = A.simulate_kappa(0.6, 0.7, n)
    print(n, "items: 95% of measured kappas in", (round(A.percentile(ks, .025), 2), round(A.percentile(ks, .975), 2)))
ks = A.simulate_kappa(0.6, 0.7, 50)
assert (round(A.percentile(ks, .025), 2), round(A.percentile(ks, .975), 2)) == (0.33, 0.82)